# 在语义内核中使用代理草稿板进行聊天历史缩减

本笔记本演示如何使用语义内核的聊天历史缩减功能以及代理草稿板来在对话中维护上下文。这对于构建能够处理长对话而不超出token限制的高效AI代理至关重要。

## 你将学到:
1. **聊天历史缩减**: 如何自动总结对话历史以管理token使用
2. **代理草稿板**: 用于跟踪用户偏好和已完成任务的持久化记忆系统
3. **Token使用跟踪**: 监控使用和不使用历史缩减时的token使用变化

## 前置条件:
- 配置了环境变量的Azure OpenAI设置
- 对之前课程中基本代理概念的理解

## 导入所需包

In [1]:
# 导入必要的包
import json
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display, HTML, Markdown
from typing import Annotated, Optional

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.completion_usage import CompletionUsage
from semantic_kernel.contents import ChatHistorySummarizationReducer
from semantic_kernel.functions import kernel_function

## 理解代理草稿板

### 什么是代理草稿板?

一个**代理草稿板**是一个持久化记忆系统,代理用它来:
- **跟踪已完成任务**: 记录已为用户完成的工作
- **存储用户偏好**: 记住喜好、厌恶和要求
- **维护上下文**: 在对话中保持重要信息的可访问性
- **减少冗余**: 避免重复询问相同的问题

### 它如何工作:
1. **写操作**: 代理在学习新信息后更新草稿板
2. **读操作**: 代理在做出决策时查阅草稿板
3. **持久化**: 即使聊天历史被缩减,信息仍然保留

可以把它看作是代理的个人笔记本,它补充了对话历史。

## 环境配置

In [2]:
# 加载环境变量
load_dotenv()

# 创建Azure OpenAI服务
# 从 .env 读取 MODEL_FREE_8B、API_URL、API_KEY
MODEL_FREE_8B = os.getenv("MODEL_FREE_8B")
API_URL = os.getenv("API_URL")
API_KEY = os.getenv("API_KEY")

chat_service = AzureChatCompletion(
    deployment_name=MODEL_FREE_8B,
    endpoint=API_URL,
    api_key=API_KEY,
)

print("✅ Azure OpenAI服务已配置")

✅ Azure OpenAI服务已配置


## 创建代理草稿板插件

此插件允许代理读取和写入持久化的草稿板文件。

In [3]:
class ScratchpadPlugin:
    """用于管理代理草稿板的插件 - 用于用户偏好和已完成任务的持久化记忆"""
    
    def __init__(self, filepath: str = "agent_scratchpad.md"):
        self.filepath = Path(filepath)
        # 如果草稿板不存在则初始化
        if not self.filepath.exists():
            self.filepath.write_text("# Agent Scratchpad\n\n## User Preferences\n\n## Completed Tasks\n\n")
    
    @kernel_function(
        description="读取当前代理草稿板以获取用户的旅行偏好和已完成任务"
    )
    def read_scratchpad(self) -> Annotated[str, "代理草稿板的内容"]:
        """读取当前草稿板内容"""
        return self.filepath.read_text()
    
    @kernel_function(
        description="使用新的用户旅行偏好或已完成任务更新代理草稿板"
    )
    def update_scratchpad(
        self,
        category: Annotated[str, "要更新的类别: 'preferences' 或 'tasks'"],
        content: Annotated[str, "要添加的新内容"]
    ) -> Annotated[str, "更新的确认信息"]:
        """用新信息更新草稿板"""
        current_content = self.filepath.read_text()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        if category.lower() == "preferences":
            # 找到偏好部分并追加
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## User Preferences" in line:
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        elif category.lower() == "tasks":
            # 找到任务部分并追加
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## Completed Tasks" in line:
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        
        self.filepath.write_text(current_content)
        return f"✅ 草稿板已更新 {category}: {content}"

# 创建草稿板插件
scratchpad_plugin = ScratchpadPlugin("vacation_agent_scratchpad.md")
print("📝 草稿板插件已创建")

📝 草稿板插件已创建


## 初始化聊天历史缩减器

当聊天历史超过阈值时,ChatHistorySummarizationReducer会自动总结对话历史。

In [4]:
# 配置缩减参数
REDUCER_TARGET_COUNT = 5  # 缩减后保留的目标消息数
REDUCER_THRESHOLD = 15    # 当消息数超过此值时触发缩减

# 创建历史总结缩减器
history_reducer = ChatHistorySummarizationReducer(
    service=chat_service,
    target_count=REDUCER_TARGET_COUNT,
    threshold_count=REDUCER_THRESHOLD,
)

print(f"🔄 聊天历史缩减器已配置:")
print(f"   - 缩减触发阈值: {REDUCER_THRESHOLD} 条消息")
print(f"   - 缩减后保留: {REDUCER_TARGET_COUNT} 条消息")

🔄 聊天历史缩减器已配置:
   - 缩减触发阈值: 15 条消息
   - 缩减后保留: 5 条消息


## 创建度假规划代理

此代理将通过草稿板维护上下文来帮助用户规划度假。

In [5]:
# 创建带有详细指令的度假规划代理
agent = ChatCompletionAgent(
    service=chat_service,
    name="VacationPlannerAgent",
    instructions="""
    你是一个有用的度假规划助手。你的工作是帮助用户规划他们的完美度假。
    
    关键草稿板规则 - 你必须遵循这些:
    1. 首要行动: 开始任何对话时,立即调用 read_scratchpad() 检查现有偏好
    2. 学习偏好后: 当用户提到任何偏好(目的地、活动、预算、日期)时,
       立即调用 update_scratchpad(),类别为 'preferences'
    3. 完成任务后: 当你完成创建行程或完成任何任务时,
       立即调用 update_scratchpad(),类别为 'tasks'
    4. 新行程前: 创建任何行程前总是调用 read_scratchpad()
    
    更新草稿板的示例:
    - 用户说"我喜欢海滩" → update_scratchpad('preferences', '喜欢海滩目的地')
    - 用户说"预算是3000美元" → update_scratchpad('preferences', '预算: 每人每周3000美元')
    - 你创建行程 → update_scratchpad('tasks', '为海滩度假创建了巴厘岛行程')
    
    规划过程:
    1. 首先读取草稿板
    2. 如果未找到偏好,询问偏好
    3. 用新信息更新草稿板
    4. 创建详细行程
    5. 用已完成任务更新草稿板
    
    明确说明: 总是宣布你何时检查或更新草稿板。
    """,
    plugins=[scratchpad_plugin],
)

print("🤖 度假规划代理已创建,带有增强的草稿板指令")

🤖 度假规划代理已创建,带有增强的草稿板指令


## 用于显示和Token跟踪的辅助函数

In [6]:
# Token跟踪类
class TokenTracker:
    def __init__(self):
        self.history = []
        self.total_usage = CompletionUsage()
        self.reduction_events = []  # 跟踪何时发生缩减

    def add_usage(self, usage: CompletionUsage, message_num: int, thread_length: int = None):
        if usage:
            self.total_usage += usage
            entry = {
                "message_num": message_num,
                "prompt_tokens": usage.prompt_tokens,
                "completion_tokens": usage.completion_tokens,
                "total_tokens": usage.prompt_tokens + usage.completion_tokens,
                "cumulative_tokens": self.total_usage.prompt_tokens + self.total_usage.completion_tokens,
                "thread_length": thread_length
            }
            self.history.append(entry)

    def mark_reduction(self, message_num: int):
        self.reduction_events.append(message_num)

    def display_chart(self):
        """显示每条消息的token使用情况以及缩减的影响"""
        if not self.history:
            return

        html = "<div style='font-family: monospace; background: #2d2d2d; color: #f0f0f0; padding: 15px; border-radius: 8px; border: 1px solid #444;'>"
        html += "<h4 style='color: #4fc3f7; margin-top: 0;'>📊 Token使用分析</h4>"
        html += "<pre style='color: #f0f0f0; margin: 0;'>"

        # 显示每条消息的prompt tokens以查看缩减影响
        html += "<span style='color: #81c784;'>每条消息的Prompt Tokens(显示对话上下文大小):</span>\n"
        max_prompt = max(h["prompt_tokens"] for h in self.history)
        scale = 50 / max_prompt if max_prompt > 0 else 1

        for i, h in enumerate(self.history):
            bar_length = int(h["prompt_tokens"] * scale)
            bar = "█" * bar_length
            reduction_marker = " <span style='color: #ff6b6b;'>← 缩减!</span>" if h[
                "message_num"] in self.reduction_events else ""
            html += f"<span style='color: #aaa;'>消息 {h['message_num']:2d}:</span> <span style='color: #4fc3f7;'>{bar}</span> <span style='color: #ffd93d;'>{h['prompt_tokens']:,} tokens</span>{reduction_marker}\n"

        html += "\n</pre></div>"
        display(HTML(html))

        # 计算缩减影响
        if self.reduction_events:
            # 找到第一次缩减前后的消息
            first_reduction_msg = self.reduction_events[0]
            before_reduction = None
            after_reduction = None

            for h in self.history:
                if h["message_num"] == first_reduction_msg - 1:
                    before_reduction = h["prompt_tokens"]
                elif h["message_num"] == first_reduction_msg:
                    after_reduction = h["prompt_tokens"]

            if before_reduction and after_reduction:
                reduction_amount = before_reduction - after_reduction
                reduction_percent = (reduction_amount / before_reduction * 100)
                print(f"\n🔄 实际缩减影响:")
                print(f"缩减前的prompt tokens: {before_reduction:,}")
                print(f"缩减后的prompt tokens: {after_reduction:,}")
                print(
                    f"节省的tokens: {reduction_amount:,} ({reduction_percent:.1f}%)")

# 用于清晰输出的显示函数


def display_message(role: str, content: str, color: str = "#2E8B57"):
    """以良好的格式显示消息,适用于浅色和深色主题"""
        # 使用适应主题的半透明背景
    html = f"""
    <div style='
        margin: 10px 0; 
        padding: 12px 15px; 
        border-left: 4px solid {color}; 
        background: rgba(128, 128, 128, 0.1); 
        border-radius: 4px;
        color: inherit;
    '>
        <strong style='color: {color}; font-size: 14px;'>{role}:</strong><br>
        <div style='margin-top: 8px; white-space: pre-wrap; color: inherit; font-size: 14px;'>{content}</div>
    </div>
    """
    display(HTML(html))


# 初始化token跟踪器
token_tracker = TokenTracker()
print("📊 Token跟踪已初始化")

📊 Token跟踪已初始化


## 运行度假规划对话

现在让我们运行一个完整的对话演示:
1. 初始规划请求
2. 偏好收集
3. 行程创建
4. 地点变更
5. 聊天历史缩减
6. 草稿板使用

In [8]:
# Define the conversation flow
user_inputs = [
    "I'm thinking about planning a vacation. Can you help me?",
    "I love beach destinations with great food and culture. I enjoy water sports, exploring local markets, and trying authentic cuisine. My budget is around $3000 per person for a week.",
    "That sounds perfect! Please create a detailed itinerary for Bali.",
    "Actually, I've changed my mind. I'd prefer to go to the Greek islands instead. Can you create a new itinerary?",
    "What's the weather like there?",
    "What should I pack?",
    "Are there any cultural customs I should know about?",
    "What's the best way to get around?"
]


async def run_vacation_planning():
    """Run the vacation planning conversation with token tracking and history reduction"""

    # Create thread with history reducer
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    message_count = 0
    scratchpad_operations = 0  # Track scratchpad usage

    print("🚀 Starting Vacation Planning Session\n")

    # Process conversation
    for i, user_input in enumerate(user_inputs):
        message_count += 1
        display_message("User", user_input, "#4fc3f7")  # Blue for user

        # Get agent response
        full_response = ""
        usage = None
        function_calls = []  # Track function calls

        async for response in agent.invoke(
            messages=user_input,
            thread=thread,
        ):
            if response.content:
                full_response += str(response.content)
            if response.metadata.get("usage"):
                usage = response.metadata["usage"]
            thread = response.thread

        display_message(f"{agent.name}", full_response,
                        "#81c784")  # Green for agent

        # Track tokens with thread length
        if usage:
            token_tracker.add_usage(usage, message_count, len(thread))

        # Check thread status and look for scratchpad operations
        print(f"📝 Thread has {len(thread)} messages")

        # Count scratchpad operations in this turn
        turn_scratchpad_ops = 0
        async for msg in thread.get_messages():
            if hasattr(msg, 'content') and msg.content:
                content_str = str(msg.content)
                if 'read_scratchpad' in content_str or 'update_scratchpad' in content_str:
                    turn_scratchpad_ops += 1

        if turn_scratchpad_ops > scratchpad_operations:
            print(
                f"   📝 Scratchpad operations detected: {turn_scratchpad_ops - scratchpad_operations} new operations")
            scratchpad_operations = turn_scratchpad_ops

        # Show message types for first message
        if i == 0:
            message_types = []
            async for msg in thread.get_messages():
                msg_type = msg.role.value if hasattr(
                    msg.role, 'value') else str(msg.role)
                message_types.append(msg_type)
            print(f"   Message types: {message_types[:10]}..." if len(
                message_types) > 10 else f"   Message types: {message_types}")

        # Check if reduction should happen
        if len(thread) > REDUCER_THRESHOLD:
            print(
                f"   ⚠️ Thread length ({len(thread)}) exceeds threshold ({REDUCER_THRESHOLD})")

            # Attempt reduction
            is_reduced = await thread.reduce()
            if is_reduced:
                print(
                    f"\n🔄 HISTORY REDUCED! Thread now has {len(thread)} messages\n")
                token_tracker.mark_reduction(message_count + 1)

                # Show summary if available
                async for msg in thread.get_messages():
                    if msg.metadata and msg.metadata.get("__summary__"):
                        display_message("System Summary", str(
                            msg.content), "#ff6b6b")
                        break

    # Display final token usage chart
    print("\n--- Token Usage Analysis ---")
    token_tracker.display_chart()

    # Show final scratchpad contents
    print("\n--- Final Scratchpad Contents ---")
    scratchpad_contents = scratchpad_plugin.read_scratchpad()
    display(Markdown(scratchpad_contents))

    print(f"\n📊 Total scratchpad operations: {scratchpad_operations}")

    return thread

# Run the conversation
thread = await run_vacation_planning()

🚀 Starting Vacation Planning Session



ServiceResponseException: ("<class 'semantic_kernel.connectors.ai.open_ai.services.azure_chat_completion.AzureChatCompletion'> service failed to complete the prompt", NotFoundError('Not Found'))